# 03 — Carga y búsqueda en Qdrant
Adaptado del notebook grupal. Usa el puerto 6333 y una colección exclusiva de esta prueba local. No modifica PostgreSQL.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

carpeta = Path("../data/processed")
df = pd.concat(
    [pd.read_csv(f) for f in sorted(carpeta.glob("chunks_*.csv"))],
    ignore_index=True,
)
vectores = np.load(carpeta / "embeddings.npy")

# Control: debe haber un vector por cada fragmento
assert len(df) == len(vectores), "No coinciden: vuelve a correr el notebook 02"

print("fragmentos:", len(df), "| vectores:", vectores.shape)
assert len(df) >= 10 and vectores.shape == (len(df), 384)

fragmentos: 20 | vectores: (20, 384)


In [11]:
from qdrant_client import QdrantClient, models
cliente = QdrantClient(host='localhost', port=6333, prefer_grpc=False)
nombre_coleccion = 'bdii_decalogo_qdrant'
# Reinicia exclusivamente la colección dedicada a esta prueba.
if cliente.collection_exists(nombre_coleccion):
    cliente.delete_collection(nombre_coleccion)
cliente.create_collection(
    collection_name=nombre_coleccion,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
    hnsw_config=models.HnswConfigDiff(m=16, ef_construct=64)
)
print('Colección lista')

Colección lista


In [12]:
puntos = [models.PointStruct(
    id=i+1, vector=vectores[i].tolist(),
    payload={"contenido": fila.contenido, "documento": fila.documento,
             "categoria": fila.categoria, "fuente": fila.fuente,
             "seccion": fila.seccion, "numero_item": int(fila.numero_item)}
) for i, fila in enumerate(df.itertuples())]
cliente.upsert(collection_name=nombre_coleccion, points=puntos, wait=True)
total = cliente.count(collection_name=nombre_coleccion, exact=True).count
assert total == len(df)
print('Fragmentos en Qdrant:', total)

Fragmentos en Qdrant: 20


In [13]:
from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
def buscar_qdrant(pregunta, k=3):
    v = modelo.encode(pregunta).tolist()
    puntos = cliente.query_points(collection_name=nombre_coleccion, query=v,
        limit=k, with_payload=True, with_vectors=False).points
    return [(p.payload['contenido'], p.score) for p in puntos]
for texto, score in buscar_qdrant('puedo consultar gratis mi historial crediticio'):
    print(f'Similitud: {score:.4f}\n{texto}\n')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2012.08it/s]


Similitud: 0.4339
Obligaciones de las instituciones supervisadas por la CNBS: Permitir al usuario la consulta gratuita y por los canales que la institución supervisada establezca, en relación a: Estado de sus cuentas, créditos, servicios y productos, incluyendo la información que la institución supervisada reporta a la Central de Información Crediticia (CIC).

Similitud: 0.3872
Deberes y buenas prácticas del usuario financiero: Evitar sobre endeudarse con las instituciones supervisadas y con particulares, velando por la economía y bienestar financiero del hogar.

Similitud: 0.3832
Obligaciones de las instituciones supervisadas por la CNBS: No requerir nuevamente al usuario financiero, información que previamente fue entregada a la institución supervisada, sin perjuicio de la información del usuario financiero que deba mantenerse actualizada.



In [21]:
import time

preguntas = [
    "puedo consultar gratis mi historial crediticio",
    "que debe hacer el usuario antes de firmar un contrato",
    "la institucion debe atender mis reclamos",
    "como evitar el sobreendeudamiento",
    "que informacion debe darme el banco sobre mis cuentas",
    "quien supervisa a las instituciones financieras",
    "puedo pedir una copia de mi contrato",
    "como proteger mis datos personales",
    "que hago si cobran comisiones no informadas",
    "derechos del usuario financiero",
]

vectores_preg = {p: modelo.encode(p).tolist() for p in preguntas}

def consulta_qdrant(v, k):
    return cliente.query_points(
        collection_name=nombre_coleccion, query=v, limit=k,
        with_payload=False, with_vectors=False
    ).points

# Calentamiento: consultas que no se miden
for v in list(vectores_preg.values())[:3]:
    consulta_qdrant(v, 3)

filas_tiempos = []
for k in (3, 5, 10):
    for p, v in vectores_preg.items():
        t0 = time.perf_counter()
        consulta_qdrant(v, k)
        ms = (time.perf_counter() - t0) * 1000
        filas_tiempos.append({"base": "qdrant", "k": k, "pregunta": p, "ms": ms})

tiempos = pd.DataFrame(filas_tiempos)
tiempos.to_csv("../data/processed/tiempos_qdrant.csv", index=False)

print(tiempos.groupby("k")["ms"].agg(["min", "max", "mean"]).round(3))

      min     max   mean
k                       
3   5.157   6.263  5.646
5   5.582  23.906  8.040
10  5.029   7.885  5.821


Nota: se mide la latencia desde el cliente Python, incluyendo transporte y decodificación, no la generación de embeddings. Con 20 fragmentos Qdrant puede usar búsqueda plana en lugar de HNSW; configurar HNSW no prueba que se utilice.